# Função Plenóptica interativa


### Importações

In [2]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import animation
from matplotlib.gridspec import GridSpec
from mpl_toolkits.mplot3d import Axes3D
from IPython.display import HTML, display
import ipywidgets as widgets

%matplotlib inline

try:
    import google.colab
    from google.colab import output as _colab_output
    _colab_output.enable_custom_widget_manager()
except Exception:
    pass

ACCENT = "#ffd166"
plt.rcParams.update({
    "figure.facecolor": "#0e1117", "axes.facecolor":   "#0e1117",
    "savefig.facecolor": "#0e1117", "text.color":       "#e6e6e6",
    "axes.labelcolor":   "#cfd2d6", "xtick.color":      "#9aa0a6",
    "ytick.color":       "#9aa0a6", "axes.edgecolor":   "#3a3f4b",
    "axes.titlecolor":   "#f2f2f2", "font.size": 10.5, "figure.dpi": 110,
})

### Motor da função plenóptica

In [3]:
def normalize(v, axis=-1):
    n = np.linalg.norm(v, axis=axis, keepdims=True)
    return v / np.maximum(n, 1e-12)

SKY_TOP     = np.array([0.04, 0.06, 0.15])
SKY_HORIZON = np.array([0.55, 0.63, 0.80])
SUN_DIR     = normalize(np.array([-0.45, 0.70, -0.30]))
FLOOR_C1    = np.array([0.20, 0.22, 0.27])
FLOOR_C2    = np.array([0.30, 0.33, 0.40])
FLOOR_TILE  = 2.0
FOG_DIST    = 42.0

_STATIC = [
    ( 6.0, 1.0,  2.0, 1.0, 0.86, 0.22, 0.20, 0.5),
    (-5.5, 1.6,  5.0, 1.6, 0.20, 0.42, 0.85, 0.4),
    ( 2.5, 0.8, -7.0, 0.8, 0.25, 0.75, 0.38, 0.5),
    (-7.5, 2.0, -3.5, 2.0, 0.92, 0.74, 0.22, 0.85),
    ( 8.5, 0.6, -6.0, 0.6, 0.93, 0.93, 0.96, 0.7),
    ( 0.0, 1.3,  9.5, 1.3, 0.60, 0.30, 0.82, 0.5),
]

def scene_at(t):
    arr     = np.array(_STATIC, float)
    centers = arr[:, :3].copy()
    radii   = arr[:, 3].copy()
    albedo  = arr[:, 4:7].copy()
    spec    = arr[:, 7].copy()
    ang     = 2 * np.pi * t
    orbit_r = 4.2
    dyn_c   = np.array([orbit_r*np.cos(ang),
                        1.1 + 0.6*np.sin(2*np.pi*t),
                        orbit_r*np.sin(ang)])
    centers = np.vstack([centers, dyn_c])
    radii   = np.append(radii, 0.9)
    albedo  = np.vstack([albedo, [0.13, 0.82, 0.78]])
    spec    = np.append(spec, 0.6)
    return centers, radii, albedo, spec

def ray_scene(O, D, t=0.0):
    O = np.asarray(O, float)
    N = D.shape[0]
    centers, radii, albedo, spec = scene_at(t)
    M = centers.shape[0]

    t_hit = np.full(N, np.inf)
    alb   = np.zeros((N, 3))
    nrm   = np.zeros((N, 3))
    spc   = np.zeros(N)
    hit   = np.zeros(N, bool)

    #piso y=0
    dy = D[:, 1]
    valid = dy < -1e-6
    tp = np.full(N, np.inf)
    tp[valid] = -O[1] / dy[valid]
    tp[tp <= 1e-4] = np.inf
    upd = tp < t_hit
    if upd.any():
        P = O[None, :] + tp[upd, None] * D[upd]
        chk = (np.floor(P[:, 0]/FLOOR_TILE) + np.floor(P[:, 2]/FLOOR_TILE)).astype(int)
        c = np.where((chk % 2 == 0)[:, None], FLOOR_C1[None, :], FLOOR_C2[None, :])
        t_hit[upd] = tp[upd]; alb[upd] = c
        nrm[upd] = np.array([0., 1., 0.]); spc[upd] = 0.04; hit[upd] = True

    #esferas
    for m in range(M):
        oc = O - centers[m]
        h  = D @ oc
        c0 = oc @ oc - radii[m] ** 2
        disc = h*h - c0
        ok = disc > 0
        if not ok.any():
            continue
        sq = np.sqrt(disc[ok])
        tn = -h[ok] - sq
        tf = -h[ok] + sq
        tc = np.where(tn > 1e-4, tn, tf)
        good = tc > 1e-4
        idx = np.where(ok)[0][good]; tc = tc[good]
        better = tc < t_hit[idx]
        idx = idx[better]; tc = tc[better]
        if idx.size:
            P = O[None, :] + tc[:, None] * D[idx]
            n = normalize(P - centers[m][None, :])
            t_hit[idx] = tc; alb[idx] = albedo[m]
            nrm[idx] = n; spc[idx] = spec[m]; hit[idx] = True

    #sombreamento
    col = np.zeros((N, 3))
    miss = ~hit
    if miss.any():
        up = np.clip(D[miss, 1], 0, 1)
        sky = (1-up)[:, None]*SKY_HORIZON[None, :] + up[:, None]*SKY_TOP[None, :]
        sun = np.clip(D[miss] @ SUN_DIR, 0, 1) ** 90
        sky = sky + 0.9*sun[:, None]*np.array([1.0, 0.95, 0.8])[None, :]
        col[miss] = sky
    if hit.any():
        Dh, Nh, Ah, Sh = D[hit], nrm[hit], alb[hit], spc[hit]
        ndl = np.clip(Nh @ SUN_DIR, 0, 1)
        Hh  = normalize(-Dh + SUN_DIR[None, :])
        ndh = np.clip(np.sum(Nh*Hh, axis=1), 0, 1)
        specular = (ndh ** 48) * Sh
        shade = (0.25 + 0.95*ndl)[:, None]*Ah + specular[:, None]*np.ones(3)[None, :]
        fog = np.clip(t_hit[hit]/FOG_DIST, 0, 1)[:, None]
        shade = (1-fog)*shade + fog*SKY_HORIZON[None, :]
        col[hit] = shade

    return np.clip(col, 0, 1), t_hit

def dirs_cyl(W, H, vfov_deg):
    hmax = np.tan(np.radians(vfov_deg/2))
    th = np.linspace(0, 2*np.pi, W, endpoint=False)
    h  = np.linspace(hmax, -hmax, H)
    TH, Hh = np.meshgrid(th, h)
    D = np.stack([np.sin(TH), Hh, np.cos(TH)], -1).reshape(-1, 3)
    return normalize(D)

def dirs_sph(W, H, vfov_deg):
    th = np.linspace(0, 2*np.pi, W, endpoint=False)
    half = np.radians(vfov_deg/2)
    ph = np.linspace(half, -half, H)
    TH, PH = np.meshgrid(th, ph)
    D = np.stack([np.cos(PH)*np.sin(TH), np.sin(PH), np.cos(PH)*np.cos(TH)], -1).reshape(-1, 3)
    return normalize(D)

def render_panorama(V, W=320, H=140, t=0.0, proj="cylindrical", vfov_deg=110):
    D = dirs_cyl(W, H, vfov_deg) if proj == "cylindrical" else dirs_sph(W, H, vfov_deg)
    col, depth = ray_scene(V, D, t)
    return col.reshape(H, W, 3), depth.reshape(H, W)

## Interface interativa

In [4]:
RAY_AZ = np.array([10,55,120,205,300], float)  #graus

def cast_single(V,az_deg,t):
    a=np.radians(az_deg); d=np.array([np.sin(a),0.0,np.cos(a)])
    col,dep=ray_scene(V,d[None,:],t)
    return d,float(dep[0]),col[0]

def draw_topdown(ax,V,t):
    centers,radii,alb,spec=scene_at(t)
    ax.set_facecolor("#0e1117")
    for c,r,a in zip(centers,radii,alb):
        ax.add_patch(plt.Circle((c[0],c[2]),r,color=a,alpha=0.9,ec="white",lw=0.5))
    for az in RAY_AZ:
        d,dep,col=cast_single(V,az,t)
        L=dep if np.isfinite(dep) else 30
        ax.plot([V[0],V[0]+d[0]*L],[V[2],V[2]+d[2]*L],color=col,lw=1.6,alpha=0.9)
    ax.plot(V[0],V[2],marker="o",ms=11,mfc="#ffd166",mec="black",mew=1.2,zorder=5)
    ax.add_patch(plt.Rectangle((-3,-3),6,6,fill=False,ec="#4cc9f0",ls="--",lw=1,alpha=.7))
    ax.set_xlim(-13,13); ax.set_ylim(-13,13); ax.set_aspect("equal")
    ax.set_title("Mapa (plano XZ) — observador e raios"); ax.set_xlabel("X"); ax.set_ylabel("Z")

def draw_scene3d(ax,V,t):
    centers,radii,alb,spec=scene_at(t)
    u=np.linspace(0,2*np.pi,14); v=np.linspace(0,np.pi,8)
    su=np.outer(np.cos(u),np.sin(v)); sv=np.outer(np.sin(u),np.sin(v)); sw=np.outer(np.ones_like(u),np.cos(v))
    for c,r,a in zip(centers,radii,alb):
        ax.plot_surface(c[0]+r*su,c[2]+r*sv,c[1]+r*sw,color=a,alpha=0.85,linewidth=0,shade=True)
    g=np.linspace(-12,12,2); gx,gz=np.meshgrid(g,g)
    ax.plot_surface(gx,gz,np.zeros_like(gx),color="#222831",alpha=0.4)
    for az in RAY_AZ:
        d,dep,col=cast_single(V,az,t); L=dep if np.isfinite(dep) else 25
        ax.plot([V[0],V[0]+d[0]*L],[V[2],V[2]+d[2]*L],[V[1],V[1]+d[1]*L],color=col,lw=1.5)
    ax.scatter([V[0]],[V[2]],[V[1]],c="#ffd166",s=60,ec="black",depthshade=False)
    ax.set_xlim(-12,12); ax.set_ylim(-12,12); ax.set_zlim(0,7)
    ax.set_box_aspect((1,1,0.35)); ax.set_title("Cena 3D + observador")
    ax.set_xlabel("X"); ax.set_ylabel("Z"); ax.set_zlabel("Y")
    ax.view_init(elev=22,azim=-60)
    ax.xaxis.pane.fill=False; ax.yaxis.pane.fill=False; ax.zaxis.pane.fill=False

def draw_pano(ax,img,V,t,proj,vfov):
    H,W,_=img.shape
    ax.imshow(img,extent=[0,360,-vfov/2,vfov/2],aspect="auto",origin="upper")
    for az in RAY_AZ:
        ax.axvline(az,color="white",ls=":",lw=0.8,alpha=0.7)
    ax.set_title("Panorama angular — P(θ,φ | V, t)  [%s]"%proj)
    ax.set_xlabel("azimute θ (graus)"); ax.set_ylabel("elevação (graus)")
    ax.set_xticks(range(0,361,45))

In [5]:
_RES = {"rapida (200x84)": (200, 84), "media (260x112)": (260, 112), "alta (340x148)": (340, 148)}

def plenoptic_explorer(Vx=0.6, Vz=-0.4, Vy=1.3, t=0.15,
                       proj="cylindrical", fov=110, res="media (260x112)"):
    V = np.array([Vx, Vy, Vz]); W, H = _RES[res]
    img, _ = render_panorama(V, W, H, t, proj, fov)
    fig = plt.figure(figsize=(15, 7))
    gs = GridSpec(2, 2, figure=fig, width_ratios=[1, 1.25],
                  height_ratios=[1, 1], hspace=0.32, wspace=0.18)
    ax3d = fig.add_subplot(gs[0, 0], projection="3d"); draw_scene3d(ax3d, V, t)
    axtd = fig.add_subplot(gs[1, 0]); draw_topdown(axtd, V, t)
    axpn = fig.add_subplot(gs[:, 1]); draw_pano(axpn, img, V, t, proj, fov)
    plt.show()

ui = widgets.interactive(
    plenoptic_explorer,
    Vx=widgets.FloatSlider(min=-3, max=3, step=0.2, value=0.6,
                           continuous_update=False, description="Vx"),
    Vz=widgets.FloatSlider(min=-3, max=3, step=0.2, value=-0.4,
                           continuous_update=False, description="Vz"),
    Vy=widgets.FloatSlider(min=0.4, max=3.0, step=0.1, value=1.3,
                           continuous_update=False, description="Vy (altura)"),
    t=widgets.FloatSlider(min=0.0, max=1.0, step=0.02, value=0.15,
                          continuous_update=False, description="tempo t"),
    proj=widgets.Dropdown(options=["cylindrical", "spherical"],
                          value="cylindrical", description="projecao"),
    fov=widgets.IntSlider(min=60, max=170, step=10, value=110,
                          continuous_update=False, description="FOV vert."),
    res=widgets.Dropdown(options=list(_RES.keys()),
                         value="media (260x112)", description="resolucao"),
)
display(ui)

interactive(children=(FloatSlider(value=0.6, continuous_update=False, description='Vx', max=3.0, min=-3.0, ste…

## Animação de navegação

In [6]:
NF = 24
ang = np.linspace(0, 2*np.pi, NF, endpoint=False)
path = np.stack([2.6*np.cos(ang), 1.2 + 0*ang, 2.6*np.sin(ang)], 1)
proj, vf = "cylindrical", 110; W, H = 240, 100
frames = [render_panorama(path[i], W, H, 0.0, proj, vf)[0] for i in range(NF)]

fig = plt.figure(figsize=(12, 5)); fig.patch.set_facecolor("#0e1117")
gs = fig.add_gridspec(1, 2, width_ratios=[1, 2.4], wspace=0.18)
axm = fig.add_subplot(gs[0]); axp = fig.add_subplot(gs[1])
centers, radii, alb, spec = scene_at(0.0)
for c, r, a in zip(centers, radii, alb):
    axm.add_patch(plt.Circle((c[0], c[2]), r, color=a, ec="white", lw=0.4))
axm.plot(path[:, 0], path[:, 2], color="#4cc9f0", lw=1, ls="--", alpha=0.6)
dot, = axm.plot([], [], marker="o", ms=11, mfc=ACCENT, mec="black")
axm.set_xlim(-13, 13); axm.set_ylim(-13, 13); axm.set_aspect("equal")
axm.set_facecolor("#0e1117"); axm.set_title("trajetoria do observador")
im = axp.imshow(frames[0], extent=[0, 360, -vf/2, vf/2], aspect="auto")
axp.set_title("panorama plenoptico  P(θ, φ | V(i), t)"); axp.set_xlabel("θ (graus)")
axp.set_xticks(range(0, 361, 45))

def _upd(i):
    im.set_data(frames[i]); dot.set_data([path[i, 0]], [path[i, 2]]); return im, dot

anim = animation.FuncAnimation(fig, _upd, frames=NF, interval=120, blit=True)
plt.close()
HTML(anim.to_jshtml())